# Building and Deploying a GAN Streamlit Web App on Heroku [Part 2]

Run from a fresh Python 3.11 kernel. Dependencies install in this kernel. Application files are checksum-verified from a local clone or the companion repository. All writes use an owned temporary workspace. No API keys required. The original unavailable Keras H5 models are replaced with both official CycleGAN PyTorch generators with the user’s approval. Original code remains verbatim below. CPU inference, no training. Heroku login, provisioning and deployment are manual, account-dependent steps, not executed or simulated here. No cloud app or cost is created. Local Git remotes and the actual Procfile command are tested.

In [ ]:
import os,sys,subprocess,hashlib,urllib.request,importlib.util,socket,threading,time
from pathlib import Path
from tempfile import TemporaryDirectory
from io import BytesIO
os.environ['USE_TF']='0'
os.environ['USE_FLAX']='0'
os.environ['HF_HUB_DISABLE_XET']='1'
os.environ['HF_HUB_DISABLE_PROGRESS_BARS']='1'
subprocess.run([sys.executable,'-m','pip','install','--quiet','--disable-pip-version-check', 'torch==2.10.0','torchvision==0.25.0','streamlit==1.66.1','numpy==2.3.5','pillow==12.0.0','certifi==2026.10.23'],check=True)
import certifi
os.environ['SSL_CERT_FILE']=certifi.where()
os.environ['REQUESTS_CA_BUNDLE']=certifi.where()
import numpy as np
import torch
from PIL import Image
from IPython.display import display
torch.set_num_threads(2)
workspace=TemporaryDirectory(prefix='tdb-coding09-')
work=Path(workspace.name)
files={'series/cyclegan-streamlit-deployment/runtime.py': '5163cacdd457dfa6c4eecfd048ed05348b55da641ab204dfafb659dbbc2e7bf3', 'series/cyclegan-streamlit-deployment/app.py': '1212d15d51d2227521f2b3951504a2171275bff9d51282bec1e9ed79e0927b7d', 'series/cyclegan-streamlit-deployment/requirements.txt': '7b2ed30444da253f070e3bc16de943cb89cef0271062b0d53173f7d6e6f1a3db', 'series/cyclegan-streamlit-deployment/Procfile': '08c86cded460b7cd6f4e9292040691b06f2f44a3b4c9a4aad1de5fbf7986a791', 'series/cyclegan-streamlit-deployment/.python-version': '49a506dd32096b010d75205acf3430c9ae6c40351888129499e5a5e487126c93'}
local_candidates=[p for p in [Path.cwd(),*Path.cwd().parents] if (p/'README.md').exists()]
local_repo=next((p for p in local_candidates if (p/next(iter(files))).exists()),None)
for relative,digest in files.items():
    target=work/relative;target.parent.mkdir(parents=True,exist_ok=True)
    if local_repo is not None:
        payload=(local_repo/relative).read_bytes()
    else:
        url='https://raw.githubusercontent.com/To-Data-Beyond/Generative-AI-Techanical-Tutorials/main/'+relative
        with urllib.request.urlopen(url,timeout=120)as response:payload=response.read()
    assert hashlib.sha256(payload).hexdigest()==digest,relative
    target.write_bytes(payload)
print('Python:',sys.version.split()[0],'| PyTorch:',torch.__version__)
print('Pinned dependencies and checksum-verified application files ready.')

In [ ]:
import shutil
project=work/'deploy';project.mkdir()
for filename in ['app.py','runtime.py','requirements.txt','Procfile','.python-version']:
    shutil.copy2(work/'series/cyclegan-streamlit-deployment'/filename,project/filename)
(project/'README.md').write_text('# CycleGAN Streamlit Web App\n')
def git(*args,cwd=project):
    return subprocess.run(['git','-c','user.name=Tutorial Test','-c','user.email=tutorial@example.invalid',*args],cwd=cwd,check=True,capture_output=True,text=True).stdout.strip()
git('init','-b','main');git('add','.');git('commit','-m','Add CycleGAN application')
remote=work/'local-remote.git';git('init','--bare',str(remote),cwd=work)
git('remote','add','origin',str(remote));git('push','-u','origin','main')
assert git('rev-parse','HEAD')==git('rev-parse','refs/heads/main',cwd=remote)
print('Initialized main, committed application files and pushed to a real local bare remote.')

In [ ]:
requirements=(project/'requirements.txt').read_text()
assert all('=='in line for line in requirements.splitlines()if line)
subprocess.run([sys.executable,'-m','pip','check'],check=True)
print('Pinned direct dependencies and installed dependency consistency checked.')

In [ ]:
procfile=(project/'Procfile').read_text().strip()
assert procfile=='web: streamlit run app.py --server.address=0.0.0.0 --server.port=$PORT --server.headless=true'
assert 'enableCORS=false'not in procfile
assert not(project/'setup.sh').exists()
print('Procfile binds the assigned PORT directly; no setup.sh or disabled CORS required.')

In [ ]:
assigned=socket.socket();assigned.bind(('127.0.0.1',0));port=assigned.getsockname()[1];assigned.close()
command=procfile.removeprefix('web: ').replace('$PORT',str(port)).split()
command[:1]=[sys.executable,'-m','streamlit']
env=dict(os.environ,TDB_CYCLEGAN_CACHE=str(work/'models'),STREAMLIT_BROWSER_GATHER_USAGE_STATS='false')
process=subprocess.Popen(command,cwd=project,env=env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
try:
    for attempt in range(100):
        try:
            with urllib.request.urlopen(f'http://127.0.0.1:{port}/_stcore/health',timeout=2)as response:
                if response.read()==b'ok':break
        except OSError:time.sleep(.1)
    else:raise AssertionError('Procfile process did not become healthy')
    print('Exact Procfile command starts a healthy Streamlit process on assigned PORT.')
finally:
    process.terminate();log,_=process.communicate(timeout=30)
    print('Owned process stopped; no Heroku app created.')

## Account-dependent Heroku steps (run manually)

Choose an account, app name and paid resources yourself. Check current dyno memory and build size limits before deployment. This notebook does not run these commands or claim a live deployment.

```bash
heroku login
heroku create YOUR-UNIQUE-APP-NAME
heroku git:remote -a YOUR-UNIQUE-APP-NAME
git push heroku HEAD:main
heroku ps:scale web=1
heroku logs --tail
heroku open
```

A rejected build needs diagnosis from its logs; changing a local branch is not a general fix.

In [ ]:
# A source branch must target Heroku's deployment branch explicitly.
heroku_remote=work/'heroku-remote.git';git('init','--bare',str(heroku_remote),cwd=work)
git('remote','add','heroku',str(heroku_remote));git('switch','-c','tempbranch')
git('push','heroku','HEAD:main')
assert git('rev-parse','HEAD')==git('rev-parse','refs/heads/main',cwd=heroku_remote)
assert not git('ls-tree','-r','--name-only','HEAD').endswith('.pth')
print('Alternate local branch deploys to remote main with HEAD:main; remote commit matches exactly.')
print('No authenticated Heroku login, provisioning, billing or remote deployment was performed.')
workspace.cleanup()
print('Owned local deployment-test files cleaned up.')

## Original source code

These blocks are preserved verbatim for comparison, not executed.

### Original block 1

```text
echo "# GAN-Streamlit-Web-App" >> README.md
git init
git add README.md
git commit -m "first commit"
git branch -M master
git remote add master https://github.com/USERNAME/RepName.git
git push -u origin master
```

### Original block 2

```text
pip3 freeze > requirements.txt
```

### Original block 3

```text
mkdir -p ~/.streamlit/
echo "\
[server]\n\
headless = true\n\
port = $PORT\n\
enableCORS = false\n\
\n\
" > ~/.streamlit/config.toml
```

### Original block 4

```text
web: sh setup.sh && streamlit run app.py
```

### Original block 5

```text
heroku login
```

### Original block 6

```text
''heroku: Press any key to open up the browser to login or q to exit:''
```

### Original block 7

```text
heroku create gan-streamlit-web-app
```

### Original block 8

```text
git push heroku master
```

### Original block 9

```text
! [remote rejected] master -> master (pre-receive hook declined)
error: failed to push some refs to 'https://git.heroku.com/gan-streamlit-web-app.git'
```

### Original block 10

```text
git checkout -b tempbranch
```

### Original block 11

```text
git push heroku tempbranch
```